# Workshop 4: Benchmarking Open-TYNDP Outcomes and Running CBA Workflows

:::{note} At the end of this notebook, you will be able to:

**1. Use PyPSA's statistics module to reproduce Open-TYNDP benchmarks**
  - Navigate and analyze Open-TYNDP NT scenario outcomes using PyPSA's statistics module
  - Compare modeling outcomes and methodologies interactively

**2. Use PyPSA-Explorer to investigate latest Open-TYNDP outcomes**
  - Investigate latest Open-TYNDP networks using PyPSA-Explorer and compare with benchmarking framework
  - Identify potential inconsistencies, methodological differences, areas where Open-TYNDP can improve

**3. Run CBA workflows**
  - Learn how to run CBA analysis coupled to or detached from the Scenario Building workflow
  - Modify the CBA configuration for one specific or multiple projects on one or a set of different climate years

:::

:::{note}
If you have not set up Python on your computer, you can execute this tutorial in your browser via [Google Colab](https://colab.research.google.com/). Click the rocket button in the top right corner and launch "Colab". If that doesn't work, download the `.ipynb` file and import it in [Google Colab](https://colab.research.google.com/).

Then install the required packages by uncommenting the cell below.

:::

In [ ]:
# uncomment for running this notebook on Colab
# !pip install pypsa==1.2.1 pypsa-explorer pandas matplotlib numpy pdf2image
# !apt-get install poppler-utils

In [ ]:
import os
from datetime import datetime
import pandas as pd
import pypsa
import shutil
import zipfile
from urllib.request import urlretrieve
from pdf2image import convert_from_path
from pdf2image.exceptions import PDFPageCountError
from IPython.display import SVG, Code, IFrame, Image, display
import matplotlib.pyplot as plt
from pypsa_explorer import create_app
from pathlib import Path
import cartopy.crs as ccrs
from pypsa.plot.maps.static import (
    add_legend_lines,
)

pypsa.set_option("params.statistics.nice_names", True)
pypsa.set_option("params.statistics.drop_zero", True)
pypsa.set_option("params.statistics.round", 3)
plt.rcParams["figure.figsize"] = [14, 7]
clip = 1  # TWh

In [ ]:
def unzip_with_timestamps(zip_path, extract_to, keep_zip=True):
    """Unzip a file while preserving original file timestamps."""
    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        for member in zip_ref.infolist():
            # Extract the file
            zip_ref.extract(member, extract_to)

            # Get the extracted file path
            extracted_path = os.path.join(extract_to, member.filename)

            # Get the modification time from the zip file
            date_time = datetime(*member.date_time)
            timestamp = date_time.timestamp()

            # Set both access and modification times
            os.utime(extracted_path, (timestamp, timestamp))
    if not keep_zip:
        os.remove(zip_path)

As in previous workshops, we first need to retrieve some files that we'll need today.

In [ ]:
urls = {
    "data/results-0.7.1.zip": "https://storage.googleapis.com/open-tyndp-data-store/outcomes/0.7.1/results-0.7.1.zip",
    "scripts/_helpers.py": "https://raw.githubusercontent.com/open-energy-transition/open-tyndp-workshops/refs/heads/feat/workshop-4-sb/open-tyndp-workshops/scripts/_helpers.py",
}

os.makedirs("data", exist_ok=True)
os.makedirs("scripts", exist_ok=True)
for name, url in urls.items():
    if os.path.exists(name):
        print(f"File {name} already exists. Skipping download.")
    else:
        print(f"Retrieving {name} from storage.")
        urlretrieve(url, name)
        print(f"File available in {name}.")

to_dir = "data/results-0.7.1"
if not os.path.exists(to_dir):
    print(f"Unzipping data/results-0.7.1.zip.")
    unzip_with_timestamps("data/results-0.7.1.zip", "data/results-0.7.1")
print(f"Latest NT results for Open-TYNDP v0.7.1 are available in '{to_dir}'.")

print("Done")

And we'll also import a handy helper function that we introduced last workshop to showcase benchmarking figures.

In [ ]:
from scripts._helpers import show_benchmarks

# Scenario Building

## Explore NT outcomes using the ``PyPSA.statistics`` module

`n.statistics` provides a consistent, high-level API that handles component iteration, port mapping, and carrier grouping automatically.

:::{tip}

`n.stats` is available as a shorthand alias for `n.statistics`.

:::

Each method can be called individually or explored via the summary table:

| Category | Methods |
|---|---|
| **Costs** | `capex()`, `installed_capex()`, `expanded_capex()`, `opex()`, `system_cost()` |
| **Capacity** | `installed_capacity()`, `optimal_capacity()`, `expanded_capacity()`, `capacity_factor()` |
| **Energy** | `supply()`, `withdrawal()`, `energy_balance()`, `transmission()`, `curtailment()` |
| **Market** | `prices()`, `revenue()`, `market_value()` |

Every method accepts the same filtering and grouping parameters:
| Parameter | Description |
|---|---|
| `groupby` | String, list, or callable — how to group results (default: `\"carrier\"`) |
| `groupby_method` | Aggregation function (`\"sum\"` (default), `\"mean\"`, …) |
| `groupby_time` | `\"sum\"`, `\"mean\"`, or `False` for time series — default varies by method |
| `components` | Filter to specific component types |
| `carrier` | Filter by carrier name (internal name) |
| `bus_carrier` | Filter by the carrier of the bus |
| `nice_names` | Use human-readable carrier names (default: `True`) |

:::{warning}

`prices()` has a simplified interface — `groupby` and `groupby_time` are booleans,
and it does not accept `carrier` or `components`.

:::

The full `PyPSA.statistics` API documentation is available in the [pypsa documentation](https://docs.pypsa.org/latest/user-guide/statistics/). Additionally, you can find two video tutorials on *PyPSA meets Earth*'s youtube channel ([part 1](https://www.youtube.com/watch?v=_Asjhz6We5o), [part 2](https://www.youtube.com/watch?v=nlh3azf2JJw)) for more comprehensive information and examples on how to use the statistics module. This learning material is open-source and available on [GitHub](https://github.com/open-energy-transition/pypsa-training-sessions).

### Reminder: Extracting insights from the network

Let's load the latest Open-TYNDP NT scenario outcomes (v0.7.1) again and explore them using the statistics module.

In [ ]:
# Load latest NT scenario networks
base_path = "data/results-0.7.1/NT-cy2009-20260520/networks/"


def import_network(fn: str):
    n = pypsa.Network(fn)
    n.carriers.loc["none", "color"] = "#000000"
    return n


# Load networks directly into dictionary for PyPSA-Explorer
networks = {
    "NT 2030": import_network(base_path + "base_s_all___2030.nc"),
    "NT 2040": import_network(base_path + "base_s_all___2040.nc"),
}

To start of we'll have a look at the network for the `NT 2030` scenario.

In [ ]:
scenario = "NT 2030"


For convenience, let's save the statistics accessor in a variable `s`.

In [ ]:
s = networks[scenario].stats

You can easily get a comprehensive overview of all system-level metrics at once.

In [ ]:
s()

Of course, this can be a bit difficult to grasp. So let's have a look at some specific metrics instead, for example installed renewable capacities:

In [ ]:
(
    s.installed_capacity(
        bus_carrier="AC|low voltage",
        comps="Generator",
    )
    .div(1e3)  # GW
    .round(3)
    .to_frame("Installed Capacity [GW]")
    .filter(regex="^(?!Demand).*", axis=0)
)

We can also easily look into the outputs of the model.

So, let's investigate electricity supply and demand for our NT 2030 network using the `energy_balance` method:

In [ ]:
balance = (
    s.energy_balance(
        bus_carrier=["AC", "low voltage"],
        comps=["Generator", "Link", "StorageUnit", "Load"],
        groupby=["carrier"],
        aggregate_across_components=True,
    ).div(
        1e6
    )  # TWh
    # .sort_values(ascending=True)
)

# Format output
balance = balance[
    abs(balance.values) > clip
].to_frame(  # Filter for entries > clipped value
    "Supply (+), Demand (-) [TWh]"
)
balance.style.format("{:,.2f}")  # Make style a bit prettier

We can also visualize this for a better overview:

In [ ]:
balance.plot.barh(
    figsize=(5, 12),
    xlabel="TWh",
    title=f"Electricity Energy Balance {scenario} (clipped at {clip} TWh)",
);

### Reminder: Visualizing results using `PyPSA.statistics`

As we already introduced in a previous workshop, it is actually much quicker to explore the data with plots generated using the `PyPSA.statistics` module.

For example the electricity energy balance we investigated before...

In [ ]:
fig, ax, _ = s.energy_balance.plot.bar(
    bus_carrier=["AC", "low voltage"],
    query=f"abs(value)>{clip*1e6}",
    height=6,
)
ax.set_title(f"Electricity Energy Balance {scenario} (Clipped at {clip} TWh)");

...or we can explore time series even:

In [ ]:
s.energy_balance.plot.area(
    bus_carrier=["AC", "low voltage"],
    y="value",
    x="snapshot",
    color="carrier",
    stacked=True,
    aspect=5,
    query="snapshot < '2009-03'",
);

We can easily have a closer look at the production of a specific technology and a specific country. For example wind production in the Netherlands in NT 2030.

In [ ]:
s.energy_balance.iplot.area(
    facet_row="bus_carrier",
    facet_col="country",
    y="value",
    x="snapshot",
    carrier="wind",
    color="carrier",
    query="country == 'NL'",
)

### Compare results with ``PyPSA.NetworkCollections``

Lastly, one might also want to compare results between runs or planning years. To make this a bit easier, PyPSA v1.0 introduced a feature called `Network Collections`.

Let's have a look at what this looks like. First we define a network collection (`nc`) with our previously imported result networks for NT 2030 and NT 2040.

In [ ]:
nc = pypsa.NetworkCollection(networks)
nc

As we can see, the `NetworkCollection` contains two networks, the `NT 2030` network and the `NT 2040` network. This is exactly what we wanted.

We can now use `PyPSA.statistics` accessor directly on this `NetworkCollection` instead of a single network to get the metrics for them simultaneously. 

Let's start again by defining a helper variable `sc` to make our life a bit easier going forward.

In [ ]:
sc = nc.statistics

Now, let's have a look at the electricity energy balance again but for both networks at once:

In [ ]:
ebs = (
    sc.energy_balance(
        groupby=["bus_carrier", "carrier"],
        aggregate_across_components=True,
    )
    .div(1e6)
    .unstack("network")
    .loc[["Electricity", "Electricity Low Voltage"]]
    .droplevel("bus_carrier")
    .sort_values("NT 2030", ascending=False)
)
ebs = ebs[abs(ebs["NT 2030"]) > clip]  # Filter for entries > 1 TWh
ebs.style.format("{:,.2f}")  # Make style a bit prettier

We can again create a simple plot to make this a bit easier to investigate

In [ ]:
ebs.plot.bar(
    figsize=(16, 9),
    ylabel="Supply (+), Demand (-) [TWh]",
    title=f"Electricity Energy Balance NT (clipped at {clip} TWh)",
);

Similarly, we can also look into electricity prices in the system

In [ ]:
prices = sc.prices(bus_carrier="AC", groupby=False, weighting="time").unstack("network")
prices

...and plot them for easier readability

In [ ]:
prices.plot.bar(
    figsize=(25, 4),
    edgecolor="white",
    ylabel="€/MWh",
    xlabel="Bus Carrier",
    title="Electricity Price by node [EUR/MWh]",
);

Of course `NetworkCollections` also work with `PyPSA.statistics` quick plotting API:

In [ ]:
sc.prices.plot.bar(
    bus_carrier=["AC"],
    x="name",
    y="value",
    color="network",
    stacked=True,
    aspect=5,
    nice_names=False,
    title="Electricity Price by node [EUR/MWh]",
);

## Task 1: Grow comfortable with ``PyPSA.statistics``
Familiarize yourself with the statistics module (again) and explore the latest results of Open-TYNDP using the different methods and plots introduced today.

**Hint**: You can also refer to the [introduction](#reminder-the-pypsa-statistics-module) above for more information on the different methods and parameters of ``PyPSA.statistics``.

## Task 2: Reproduce Benchmarks
If you feel comfortable using `PyPSA.statistics`, you can try to reproduce the Open-TYNDP results from the following example of our latest [benchmarking figures](https://zenodo.org/records/20303009).

In [ ]:
show_benchmarks(
    "benchmark_electricity_price_cy2009",
    [2030],
    "data/results-0.7.1/NT-cy2009-20260520/benchmarks/tyndp-2024/graphics_s_all___all_years/by_bus",
)

## Task 3 (Advanced): Investigate Outputs

(a) Can you verify that Germany is the largest *net annual importer of H2* in 2040?

**Hint:** Look for `H2 pipeline` in the energy balance.

(b) Can you verify the amount of Offshore wind generation for the Netherlands in 2040 at *19.9 TWh*?

**Hint:** The Offshore bus carrier is `AC_OH`

(c) Can you investigate the correlation between electricity mix and H2 production in Germany for the *first week of June* in 2040? What can we notice?

## Reminder: Interactive Exploration with PyPSA-Explorer

As we introduced in the last workshop, PyPSA-Explorer is an interactive dashboard for visualizing and analyzing energy system networks. It provides:
- Energy balance analysis with both time series and aggregated views
- Capacity planning visualizations by technology and region
- Economic analysis showing CAPEX/OPEX breakdowns
- Interactive geographical network maps
- Support for visualising multiple networks

:::{note}

 PyPSA-Explorer can be launched in different ways depending on your environment:

- **Local Jupyter**: Use the terminal command (recommended) or inline display
- **Google Colab**: The dashboard launches inline, embedded directly in the notebook

Follow the instructions below for your specific environment.

:::

In [ ]:
# Detect if running on Google Colab
try:
    from google.colab import output

    IN_COLAB = True
    print(f"This notebook is running on Google Colab!")
except ImportError:
    IN_COLAB = False
    print(f"This notebook is running locally !")

port = 8050

### For Local Users

If you're running locally, we **recommend** launching PyPSA-Explorer from the terminal for optimal performance:

```bash
pypsa-explorer data/results-0.7.1/NT-cy2009-20260520/networks/base_s_all___2030.nc:NT_2030 data/results-0.7.1/NT-cy2009-20260520/networks/base_s_all___2040.nc:NT_2040
```

This command opens the dashboard in your default browser at http://localhost:8050.

**Alternative**: The cell below can launch the dashboard inline within the notebook, though the terminal method provides better performance and responsiveness.

In [ ]:
# Terminal method recommended
USE_TERMINAL = True  # Change to False if you want to launch from the notebook instead

if not IN_COLAB and not USE_TERMINAL:
    # Local Jupyter: Inline display
    app = create_app(networks)
    app.run(jupyter_mode="tab", port=port, debug=False)

### For Google Colab Users

Running PyPSA-Explorer on Google Colab requires a small workaround to display the dashboard properly inside the notebook.

First, let's define a helper function to handle the setup:

In [ ]:
def run_pypsa_explorer_in_colab(networks, port):
    print("Starting PyPSA Explorer for Google Colab...")

    # Create and start the app
    app = create_app(networks)

    import threading
    import time

    def run_server():
        app.run(jupyter_mode="external", port=port, debug=False)

    server_thread = threading.Thread(target=run_server, daemon=True)
    server_thread.start()

    # Wait for server to initialize
    time.sleep(5)
    print(f"✓ Server started on port {port}")

    # Display in iframe
    output.serve_kernel_port_as_iframe(port, height=1500)

In [ ]:
if IN_COLAB:
    run_pypsa_explorer_in_colab(networks, port)

**Tip for Colab users:** To view the dashboard in fullscreen mode, click the three dots (⋮) in the top-right corner of the output cell and select **"View output fullscreen"**.

## Task 4: Navigate ``PyPSA-Explorer``
Let's look at the latest Open-TYNDP NT scenario results (v0.7.1) again and explore them interactively using the PyPSA-Explorer. Try again to find some specific insights we manually extracted in the previous section.

(a) Can you verify that Germany is the largest *net annual importer of H2* in 2040?

(b) Can you verify the amount of Offshore wind generation for the Netherlands in 2040 at *19.9 TWh*?

(c) Can you investigate the correlation between electricity mix and H2 production in Germany for the *first week of June* in 2040?

:::{note} Using the Dashboard

Once the dashboard opens, you can explore these key features:

**1. Energy Balance Tab**
   - View production, consumption, and storage patterns over time
   - Switch between time series and aggregated views
   - Filter by energy carrier (electricity, hydrogen, etc.)
   - Filter by country or region

**2. Capacity Tab**
   - Analyze installed capacities across scenarios
   - Compare capacity buildout between 2030 and 2040
   - View breakdowns by technology type and region

**3. Economics Tab**
   - Examine costs and revenues
   - Review CAPEX and OPEX breakdowns by technology
   - Compare regional cost distributions
   - Assess investment requirements

**4. Network Map**
   - Visualize the geographical network layout
   - View an interactive map with network components
   - Zoom and pan to explore specific regions

**Tip:** Use the scenario selector buttons in the top-right corner to switch between NT 2030 and NT 2040 scenarios.

:::

# Cost-Benefit Analysis

# Solutions

## Task 2: Reproduce benchmarks

In [ ]:
show_benchmarks(
    "benchmark_electricity_price_cy2009",
    [2030],
    "data/results-0.7.1/NT-cy2009-20260520/benchmarks/tyndp-2024/graphics_s_all___all_years/by_bus",
)

In [ ]:
prices = (
    s.prices(bus_carrier="AC", weighting="time").to_frame("Open-TYNDP").sort_index()
)

ax = prices.plot.bar(figsize=(16, 4), ylabel="EUR/MWh_e", xlabel="Node")

ax.set_facecolor("#e8e8e8")
ax.set_axisbelow(True)
ax.grid(True)

# Legend with average
handles, labels = ax.get_legend_handles_labels()
new_labels = [f"{label} ({prices[label].mean():.2f} EUR/MWh_e)" for label in labels]
ax.legend(handles, new_labels, loc="upper left")

ax.set_title("Electricity Price - Scenario NT - CY 2009 - Year 2030")

## Task 3: Investigate outputs

(a) Can you verify that Germany is the largest *net annual importer of H2* in 2040?

**Hint:** Look for `H2 pipeline` in the energy balance.

(b) Can you verify the amount of Offshore wind generation for the Netherlands in 2040 at *19.9 TWh*?

(c) Can you investigate the correlation between electricity mix and H2 production in Germany for the *first week of June* in 2040? What can we notice?

In [ ]:
s = networks["NT 2040"].stats

In [ ]:
# (a)
(
    s.energy_balance(
        bus_carrier="H2",
        carrier="H2 pipeline",
        aggregate_across_components=True,
        groupby=["carrier", "bus"],
    )
    .div(1e6)
    .sort_values(ascending=False)
    .to_frame("H2 Import (+), H2 Export (-) [TWh]")
    .style.format("{:,.2f}")  # Make style a bit prettier
)

In [ ]:
# (b)
(
    s.energy_balance(
        bus_carrier="AC_OH",
        aggregate_across_components=True,
        groupby=["carrier", "country"],
    )
    .div(1e6)
    .sort_values(ascending=False)
    .to_frame("Energy [TWh]")
    .loc[:, "NL", :]
)

In [ ]:
# (c)
s.energy_balance.iplot.area(
    bus_carrier=["AC", "H2"],
    y="value",
    x="snapshot",
    color="carrier",
    stacked=True,
    facet_row="bus_carrier",
    facet_col="country",
    sharex=False,
    sharey=False,
    query="snapshot < '2009-06-08' and snapshot >= '2009-06-01' and country == 'DE'",
)

# Notebook clean up

In [ ]:
# Only clean up data when running in CI environment
if os.getenv("CI"):
    rm_dir = "data/results-0.7.1"
    print(
        f"CI environment detected. Cleaning up notebook data by removing '{rm_dir}' and '{rm_dir}.zip'."
    )
    shutil.rmtree(rm_dir, ignore_errors=True)
    Path(f"{rm_dir}.zip").unlink(missing_ok=True)
else:
    print("Skipping cleanup (not in CI environment).")